# Competição · 4/4 · avaliação, gráficos, diagnóstico e ensembles

**Antes de rodar (uma vez só):** em `MyDrive/dengue_V4/` (a pasta do V4, que já tem `dados/` e `referencias_imdc3/`) suba `comp_lib.py`, `comp_modelos.py` e `comp_inla.R`, e **substitua o `v4_lib.py`** pela versão nova de `data_imdc_2026/V4/`.
Todos os resultados vão para `MyDrive/dengue_V4/resultados_competicao/`. Se a sessão cair, rode de novo: o que já foi feito é pulado.

| Notebook | Modelos | Ambiente |
|---|---|---|
| `Comp1_CPU` | climatologia, canal_endemico, AR_longo, SURGE, ARIMA_Fourier, ETS_MSTL, Prophet, LightGBM_direto, Renovacao_S, Tamanho_x_forma | CPU |
| `Comp2_INLA` | BB_INLA, GHR_INLA | CPU com bastante RAM |
| `Comp3_GPU` | XGBoost_quantis, DeepAR, NHITS, TFT, TSMixer, Chronos, AGCRN | GPU L4 |
| `Comp4_Avaliacao` | rankings, gráficos, diagnóstico, ensembles | qualquer (CPU) |

Os notebooks 1, 2 e 3 podem rodar **ao mesmo tempo**, cada um numa sessão. O 4 pode rodar a qualquer momento: ele avalia o que já existir.

Roda a qualquer momento e avalia o que já existir em `resultados_competicao/`.

- **Por UF (a tarefa oficial):** WIS relativo ao **BB oficial**, média geométrica nas 4 safras (<1 = melhor). Os estáveis do 3º IMDC ficaram em 0,87–0,92. As 28 equipes aparecem como `IMDC:`.
- **Por região de saúde:** WIS relativo à climatologia, nas 435 regiões.
- ⚠️ Compare só modelos com `temporadas = 4`.
- ⚠️ A S4 vai só até 08/03/2026 nos nossos dados.

In [ ]:
NOTEBOOK = 'avaliacao'
PACOTES = ''

# @title Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/comp', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'comp_inla.R', 'dados', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_competicao')
os.makedirs(os.path.join(PASTA_RES, 'graficos'), exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM
for m in (V, L, CM):
    importlib.reload(m)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ctx = L.Contexto(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'))
TODAS = list(range(2016, 2026))          # 2016–2021: calibração dos pesos do ensemble · 2022–2025: S1–S4
RECENTES = list(range(2020, 2026))       # para os modelos mais caros
OFICIAIS = V.TEMPORADAS_OFICIAIS
M = 500                                  # trajetórias por região
REFAZER = False

## 1. Ranking por UF

In [ ]:
PU = L.ler_resultados(PASTA_RES, 'uf')
print('Modelos com resultado:', sorted(PU.modelo.unique()))
por_uf, res_uf = L.avaliar_uf(ctx, PU, temporadas=OFICIAIS)
res_uf['origem'] = np.where(res_uf.modelo.str.startswith('IMDC:') | (res_uf.modelo == 'BB'), '3º IMDC', 'competição')
cols = ['origem', 'modelo', 'temporadas', 'R_vs_BB', 'UFs_melhor_que_BB', 'dp_logR_vs_BB', 'R_vs_climatologia',
        'WIS_log_medio', 'IC50', 'IC80', 'IC90', 'IC95']
res_uf.to_csv(os.path.join(PASTA_RES, 'ranking_uf.csv'), index=False)
por_uf.to_csv(os.path.join(PASTA_RES, 'wis_uf_temporada.csv'), index=False)
completos = res_uf[(res_uf.origem == 'competição') & (res_uf.temporadas == len(OFICIAIS))].modelo.tolist()
res_uf[cols].round(3).head(45)

In [ ]:
# @title Mapa de calor modelo × UF, estabilidade e ranking por macrorregião
import matplotlib.pyplot as plt
fig, H = L.mapa_calor(por_uf, completos, arquivo=os.path.join(PASTA_RES, 'graficos', 'mapa_calor_uf.png')); plt.show()
L.grafico_estabilidade(res_uf[res_uf.temporadas == len(OFICIAIS)], completos,
                       arquivo=os.path.join(PASTA_RES, 'graficos', 'estabilidade.png')); plt.show()
rk = L.ranking_macro(por_uf, completos)
rk.to_csv(os.path.join(PASTA_RES, 'ranking_macrorregiao.csv'), index=False)
rk[rk.posicao <= 3].pivot(index='posicao', columns='macro', values='modelo')

In [ ]:
# @title Alvos da temporada: erro no total e na semana do pico
alv, res_alv = L.alvos_temporada(ctx, PU, OFICIAIS)
res_alv.to_csv(os.path.join(PASTA_RES, 'alvos_temporada.csv'), index=False)
res_alv.sort_values('erro_abs_log_total').round(2)

## 2. Ranking por região de saúde

In [ ]:
PR = L.ler_resultados(PASTA_RES, 'regiao')
por_reg, res_reg = L.avaliar_regiao(ctx, PR, temporadas=OFICIAIS)
res_reg.to_csv(os.path.join(PASTA_RES, 'ranking_regiao.csv'), index=False)
por_reg.to_csv(os.path.join(PASTA_RES, 'wis_regiao_temporada.csv'), index=False)
res_reg[['modelo', 'temporadas', 'R_vs_climatologia', 'regioes_melhor_que_climatologia', 'dp_logR_vs_climatologia',
         'WIS_log_medio', 'IC50', 'IC90', 'IC95']].round(3)

In [ ]:
# @title Distribuição por região: log(WIS ÷ climatologia)
b = por_reg[por_reg.modelo == 'climatologia'][['season', 'unidade', 'WIS']].rename(columns={'WIS': 'Wc'})
t = por_reg[por_reg.modelo != 'climatologia'].merge(b, on=['season', 'unidade'])
t['lr'] = np.log((t.WIS + 1e-9) / (t.Wc + 1e-9))
g = t.groupby(['modelo', 'unidade']).lr.mean().reset_index()
ordem_m = list(g.groupby('modelo').lr.median().sort_values().index)
fig, ax = plt.subplots(figsize=(10, 0.35 * len(ordem_m) + 1.5))
dados_bp = [g[g.modelo == m].lr.clip(-2, 2) for m in ordem_m]
try:
    ax.boxplot(dados_bp, vert=False, tick_labels=ordem_m, showfliers=False)   # matplotlib ≥ 3.9
except TypeError:
    ax.boxplot(dados_bp, vert=False, labels=ordem_m, showfliers=False)
ax.axvline(0, ls=':', c='k'); ax.set_xlabel('log(WIS ÷ WIS da climatologia) por região (negativo = melhor)')
fig.tight_layout(); fig.savefig(os.path.join(PASTA_RES, 'graficos', 'boxplot_regioes.png'), dpi=120); plt.show()

## 3. Gráficos de previsão

In [ ]:
# @title Leque por UF: 4 melhores + climatologia, em cada safra
top = res_uf[(res_uf.origem == 'competição') & (res_uf.temporadas == len(OFICIAIS))].sort_values('R_vs_BB').modelo.head(4).tolist()
for s in OFICIAIS:
    L.grafico_leque(PU, ctx.ver_uf, list(ctx.Du.ufs), 'uf_code', top + ['climatologia'], s,
                    f'Temporada {s}/{s + 1} · UF', os.path.join(PASTA_RES, 'graficos', f'leque_uf_{s}.png')); plt.show()

In [ ]:
# @title Leque por região de saúde: 24 regiões mais populosas (melhor modelo × climatologia)
D = ctx.Dr
maiores = D.unidades[np.argsort(-D.POP[D.ultimo_obs])[:24]]
for s in OFICIAIS:
    L.grafico_leque(PR.drop(columns='uf_code'), ctx.ver_reg, list(maiores), 'unidade', [top[0], 'climatologia'], s,
                    f'Temporada {s}/{s + 1} · 24 maiores regiões de saúde',
                    os.path.join(PASTA_RES, 'graficos', f'leque_regiao_{s}.png')); plt.show()

## 4. Diagnóstico: o que melhorar em cada modelo

In [ ]:
diag = L.diagnostico(ctx, PU, por_uf, OFICIAIS)
diag = diag.merge(res_uf[['modelo', 'R_vs_BB']], on='modelo', how='left').sort_values('R_vs_BB')
diag.to_csv(os.path.join(PASTA_RES, 'diagnostico_modelos.csv'), index=False)
diag.round(2)

In [ ]:
# @title Fidelidade: nossa reimplementação × a previsão publicada pela equipe original
pares = {'BB_INLA': 'BB', 'AR_longo': 'IMDC:lncc_lncc_arp26_dengue', 'SURGE': 'IMDC:lncc_surge_model26_dengue',
         'ARIMA_Fourier': 'IMDC:emap_epidematicos_sarimax_fixed', 'Prophet': 'IMDC:emap_epidematicos_prophet_fixed',
         'GHR_INLA': 'IMDC:bsc_ghr', 'LightGBM_direto': 'IMDC:purdue_neuralearth', 'canal_endemico': 'IMDC:cornell_bentolab',
         'XGBoost_quantis': 'IMDC:emap_xgbsillas', 'DeepAR': 'IMDC:emap_lstm', 'Renovacao_S': 'IMDC:nus_nus-cerm',
         'Tamanho_x_forma': 'IMDC:ifgw_inframind-proteus', 'Chronos': 'IMDC:rki_rki_zki_ph'}
r = res_uf.set_index('modelo')
pd.DataFrame([(a, b, r.R_vs_BB.get(a), r.R_vs_BB.get(b)) for a, b in pares.items()],
             columns=['nossa versão', 'original (3º IMDC)', 'R_vs_BB nossa', 'R_vs_BB original']).round(3)

## 5. Ensembles

Entram os modelos que têm **todas** as temporadas de 2020 a 2025.
- **mediana:** mediana dos quantis entre os membros.
- **pesos:** pesos calculados só com temporadas **anteriores** à prevista, encolhidos 50% para pesos iguais.
- **top-5 da temporada anterior:** mediana dos 5 melhores (WIS em log) na temporada anterior. É uma escolha honesta, que não olha a temporada prevista.

In [ ]:
com_recentes = [m for m in PU.modelo.unique()
                if set(RECENTES) <= set(PU[PU.modelo == m].season.unique()) and not m.startswith('ENS')]
print('Membros:', com_recentes)
assert len(com_recentes) >= 2, 'Ainda não há modelos suficientes com 2020–2025: rode antes os notebooks 1 a 3.'
ens = []
for met in ('mediana', 'pesos'):
    e, pw = V.ensemble(PU, com_recentes, f'ENS_{met}', met, ctx.ver_uf, ctx.Du)
    ens.append(e)
    if len(pw):
        display(pw[pw.season.isin(OFICIAIS)].round(2))
por_todas, _ = L.avaliar_uf(ctx, PU[PU.modelo.isin(com_recentes)], incluir_ref=False)
for s in OFICIAIS:
    ant = por_todas[(por_todas.season == s - 1)]
    top5 = ant.groupby('modelo').WIS_log.mean().nsmallest(5).index.tolist()
    g = PU[(PU.season == s) & PU.modelo.isin(top5)]
    datas = np.sort(g.date.unique())
    q = np.median(np.stack([V.de_df(g[g.modelo == m], datas, ctx.Du.ufs) for m in top5]), 0)
    ens.append(V.para_df(V.arruma_quantis(q), datas, ctx.Du.ufs, 'ENS_top5_temporada_anterior', s))
    print(s, '· top-5 da temporada anterior:', top5)
E = pd.concat(ens, ignore_index=True)
E.to_parquet(os.path.join(PASTA_RES, 'ensembles_uf.parquet'), index=False)
_, res_e = L.avaliar_uf(ctx, pd.concat([PU, E]), temporadas=OFICIAIS)
res_e = res_e[res_e.temporadas == len(OFICIAIS)]
res_e.to_csv(os.path.join(PASTA_RES, 'ranking_uf_com_ensembles.csv'), index=False)
res_e[['modelo', 'R_vs_BB', 'UFs_melhor_que_BB', 'dp_logR_vs_BB', 'WIS_log_medio', 'IC50', 'IC90']].round(3).head(25)

In [ ]:
print('Arquivos em', PASTA_RES); print(sorted(os.listdir(PASTA_RES)))